# 🤗 Bloque 4: Transformers y HuggingFace

**Objetivo:** Entender la arquitectura Transformer y usar modelos preentrenados con HuggingFace para NLP.

---

## 1. ¿Qué es un Transformer?

Introducido en *"Attention Is All You Need"* (2017), el Transformer revolucionó el NLP y ahora domina también visión, audio y más.

### Componentes clave:

```
Input → Tokenización → Embeddings → Positional Encoding
      → [Encoder Blocks] → [Decoder Blocks] → Output
```

| Componente | Qué hace |
|---|---|
| **Tokenizer** | Convierte texto en IDs numéricos |
| **Embedding** | Mapea cada token a un vector denso |
| **Positional Encoding** | Añade información de posición (el orden importa) |
| **Self-Attention** | Cada token "mira" a todos los demás para entender contexto |
| **Feed Forward** | Procesa la representación con capas densas |

---

## 2. Mecanismo de Atención — intuitivamente

Frase: *"El banco está cerca del río"*

Para entender qué significa "banco", el modelo debe prestar **atención** a "río".

Self-Attention calcula, para cada token, qué peso dar a cada otro token:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q** (Query): "¿Qué estoy buscando?"
- **K** (Key): "¿Qué información tengo?"
- **V** (Value): "La información en sí"

---

## 3. HuggingFace — el ecosistema

HuggingFace es el hub central para modelos preentrenados. Sus librerías principales:
- `transformers`: modelos y tokenizers
- `datasets`: acceso a +80.000 datasets
- `evaluate`: métricas estandarizadas
- `accelerate`: entrenamiento distribuido

In [ ]:
# !pip install transformers datasets evaluate torch

## 4. Pipelines — la forma más rápida de usar modelos

In [ ]:
from transformers import pipeline

# --- 4.1 Análisis de sentimiento ---
clasificador = pipeline('sentiment-analysis')
textos = [
    "This movie was absolutely fantastic!",
    "I hated every minute of it. Terrible.",
    "It was okay, nothing special."
]
resultados = clasificador(textos)

print("=== Sentiment Analysis ===")
for texto, resultado in zip(textos, resultados):
    print(f"  '{texto[:40]}...' → {resultado['label']} ({resultado['score']:.2%})")

In [ ]:
# --- 4.2 Named Entity Recognition (NER) ---
ner = pipeline('ner', aggregation_strategy='simple')
texto = "Apple was founded by Steve Jobs in Cupertino, California."
entidades = ner(texto)

print("=== Named Entity Recognition ===")
print(f"Texto: '{texto}'")
for e in entidades:
    print(f"  {e['word']:<20} → {e['entity_group']} (score: {e['score']:.2%})")

In [ ]:
# --- 4.3 Question Answering ---
qa = pipeline('question-answering')
contexto = """
PyTorch is an open source machine learning framework based on the Torch library,
used for applications such as computer vision and natural language processing.
It was primarily developed by Meta AI Research and is used extensively in research and production.
"""
pregunta = "Who developed PyTorch?"
respuesta = qa(question=pregunta, context=contexto)

print("=== Question Answering ===")
print(f"Pregunta: {pregunta}")
print(f"Respuesta: '{respuesta['answer']}' (score: {respuesta['score']:.2%})")

## 5. Tokenizers — cómo el modelo lee el texto

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

texto = "Hello, I'm learning about Transformers!"
tokens = tokenizer(texto, return_tensors='pt')

print(f"Texto: '{texto}'")
print(f"\nInput IDs: {tokens['input_ids']}")
print(f"\nTokens decodificados: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"\nAtención mask: {tokens['attention_mask']}")
print("\n💡 [CLS] = inicio de secuencia, [SEP] = final")

## 6. Fine-tuning de un modelo preentrenado

El **fine-tuning** es tomar un modelo preentrenado (ya conoce el lenguaje) y adaptarlo a nuestra tarea específica con pocos datos.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np

# Cargar dataset pequeño de reseñas (IMDB en versión reducida)
# En producción usarías tu propio dataset
dataset = load_dataset('imdb')

# Reducir para demo rápida
small_train = dataset['train'].shuffle(seed=42).select(range(500))
small_test  = dataset['test'].shuffle(seed=42).select(range(100))

print(f"Train size: {len(small_train)}, Test size: {len(small_test)}")
print(f"\nEjemplo: '{small_train[0]['text'][:100]}...'")
print(f"Label: {small_train[0]['label']} (0=negativo, 1=positivo)")

In [ ]:
# Tokenizar el dataset
checkpoint = 'distilbert-base-uncased'  # Versión ligera de BERT
tokenizer  = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_train = small_train.map(tokenize, batched=True)
tokenized_test  = small_test.map(tokenize, batched=True)

print("Dataset tokenizado correctamente")

In [ ]:
# Cargar modelo con cabeza de clasificación
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
)

# Métricas
import evaluate
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# NOTA: El entrenamiento puede tardar varios minutos
# trainer.train()   # Descomenta para entrenar
print("Trainer configurado. Descomenta trainer.train() para ejecutar.")

## 7. Modelos populares por tarea

| Modelo | Familia | Mejor para |
|---|---|---|
| `bert-base-uncased` | BERT | Clasificación, NER, QA |
| `distilbert-base-uncased` | DistilBERT | Igual que BERT pero 40% más rápido |
| `roberta-base` | RoBERTa | Clasificación robusta |
| `gpt2` | GPT | Generación de texto |
| `t5-small` | T5 | Traducción, resumen, QA |
| `sentence-transformers/all-MiniLM-L6-v2` | SBERT | Embeddings semánticos, búsqueda |

---

## ✅ Resumen del bloque

- Entiendes la **arquitectura Transformer** y el mecanismo de atención
- Sabes usar **pipelines** de HuggingFace para tareas comunes
- Entiendes cómo funciona un **tokenizer**
- Configuraste un **fine-tuning** completo con `Trainer`
- Conoces los **modelos más populares** por tarea

---

## ➡️ Siguiente paso

Continúa con el **Bloque 5: Feature Engineering** → `05_feature_engineering.ipynb`